In [ ]:

import os
from groq import Groq
from dotenv import load_dotenv
import json
import time

# Load environment variables from .env file
load_dotenv()

print("AI Medical Query System with Groq API")

class MedicalQuerySystem:
    def __init__(self):
        """Initialize the Groq client with API key from environment"""
        self.client = Groq(
            api_key=os.environ.get("GROQ_API_KEY")
        )
        self.model_name = "llama-3.3-70b-versatile"

    def create_medical_prompt(self, question):
        """Create a structured prompt for medical questions"""
        prompt = f"""You are a medical expert AI specializing in clinical reasoning, diagnostics, and treatment planning. 
You have advanced knowledge in medical sciences and can provide detailed, accurate medical information.

Please analyze the following medical question carefully and provide a comprehensive answer with step-by-step reasoning.

Medical Question: {question}

Please structure your response as follows:
1. First, analyze the key clinical information presented
2. Consider relevant medical conditions or differential diagnoses
3. Explain the reasoning behind your conclusions
4. Provide a clear, evidence-based answer

Remember to be thorough but concise, and base your response on established medical knowledge."""

        return prompt

    def query_medical_llm(self, question, temperature=0.3, max_tokens=1500):
        """
        Query the DeepSeek R1 model through Groq API for medical questions

        Args:
            question (str): The medical question to ask
            temperature (float): Controls randomness (0.0-1.0, lower = more focused)
            max_tokens (int): Maximum tokens in response

        Returns:
            str: The model's response
        """
        try:
            # Create the medical prompt
            prompt = self.create_medical_prompt(question)

            # Make API call to Groq
            start_time = time.time()

            completion = self.client.chat.completions.create(
                model=self.model_name,
                messages=[
                    {
                        "role": "system",
                        "content": "You are an expert medical AI assistant with deep knowledge in clinical medicine, diagnostics, and treatment protocols."
                    },
                    {
                        "role": "user", 
                        "content": prompt
                    }
                ],
                temperature=temperature,
                max_tokens=max_tokens,
                top_p=0.95,
                stream=False
            )

            end_time = time.time()
            response_time = end_time - start_time

            # Extract the response
            response = completion.choices[0].message.content

            # Print inference statistics
            print(f"\n{'='*50}")
            print(f"Response Time: {response_time:.2f} seconds")
            print(f"Model Used: {self.model_name}")
            print(f"Tokens Used: {completion.usage.total_tokens if hasattr(completion, 'usage') else 'N/A'}")
            print(f"{'='*50}")

            return response

        except Exception as e:
            print(f"Error querying Groq API: {str(e)}")
            return None

    def batch_query(self, questions_list):
        """
        Process multiple medical questions in batch

        Args:
            questions_list (list): List of medical questions

        Returns:
            list: List of responses
        """
        responses = []

        for i, question in enumerate(questions_list, 1):
            print(f"\nProcessing Question {i}/{len(questions_list)}...")
            print(f"Question: {question[:100]}...")

            response = self.query_medical_llm(question)
            responses.append({
                "question": question,
                "response": response,
                "question_number": i
            })

            # Add small delay to respect API rate limits
            time.sleep(0.5)

        return responses

# Initialize the medical query system
def main():
    # Check if API key is available
    if not os.environ.get("GROQ_API_KEY"):
        print("Error: GROQ_API_KEY not found in environment variables.")
        print("Please make sure your .env file contains: GROQ_API_KEY=your_actual_api_key")
        return

    # Initialize the system
    medical_ai = MedicalQuerySystem()

    # Test questions (same as your original examples)
    test_questions = [
        """A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or
        sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
        what would cystometry most likely reveal about her residual volume and detrusor contractions?""",

        """A 59-year-old man presents with a fever, chills, night sweats, and generalized fatigue,
        and is found to have a 12 mm vegetation on the aortic valve. Blood cultures indicate gram-positive, catalase-negative,
        gamma-hemolytic cocci in chains that do not grow in a 6.5% NaCl medium.
        What is the most likely predisposing factor for this patient's condition?""",
        
        """A 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?""",
        """A patient with psoriasis was treated with systemic steroids, and upon discontinuation of the treatment, developed generalized pustules all over the body. What is the most likely cause of this condition?""",
        """A 45-year-old man has a minor cold and pain in throat what medicine should he take?"""
    ]

    print("\nTesting Medical Query System...")

    # Process each question
    for i, question in enumerate(test_questions, 1):
        print(f"\n{'#'*80}")
        print(f"MEDICAL QUESTION {i}:")
        print(f"{'#'*80}")
        print(f"\n{question}")

        print(f"\n{'*'*50}")
        print("AI MEDICAL EXPERT RESPONSE:")
        print(f"{'*'*50}")

        response = medical_ai.query_medical_llm(question)

        if response:
            print(f"\n{response}")
        else:
            print("\nFailed to get response for this question.")

        print(f"\n{'='*80}\n")

    # Demonstrate batch processing
    print("\n" + "="*80)
    print("BATCH PROCESSING EXAMPLE")
    print("="*80)

    batch_responses = medical_ai.batch_query(test_questions)

    # Save responses to JSON file (optional)
    try:
        with open('medical_responses.json', 'w') as f:
            json.dump(batch_responses, f, indent=2)
        print(f"\nResponses saved to 'medical_responses.json'")
    except Exception as e:
        print(f"\nCould not save responses: {e}")

if __name__ == "__main__":
    main()


AI Medical Query System with Groq API

Testing Medical Query System...

################################################################################
MEDICAL QUESTION 1:
################################################################################

A 61-year-old woman with a long history of involuntary urine loss during activities like coughing or
        sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these findings,
        what would cystometry most likely reveal about her residual volume and detrusor contractions?

**************************************************
AI MEDICAL EXPERT RESPONSE:
**************************************************

Response Time: 2.74 seconds
Model Used: llama-3.3-70b-versatile
Tokens Used: 899

To address the medical question presented, let's break down the analysis into the structured steps requested:

1. **Analysis of Key Clinical Information**:
   - The patient is a 61-year-old woman.
   - She experienc